In [2]:
# ruff: noqa: F401, F403

import os
import subprocess
import sys
import typing as tp

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch

from IPython.display import *

from pacer import (
    CoordinateSystem,
    # read_dat_file,
    DatVersion,
    GPMFSource,
    GPSSample,
    Lap,
    Laps,
    Point,
    PointInTime_GPSSample,
    RawGPSSource,
    Segment,
    SequentialGPSSource,
    Vec3f,
)

In [33]:
speed = 90 # kmh
speed /= 3.6 # m/s
speed * 0.1 # m per 1/10s

2.5

In [15]:
from functools import lru_cache


cs = None

@lru_cache
def locate_track(file: str) -> GPSSample:
    f = GPMFSource(file)
    samples = []

    def on_sample(s, _, _2):
        samples.append(s)

    max_samples = 1000
    while not f.is_end() and len(samples) < max_samples:
        f.read_samples(on_sample)
        f.next()

    if len(samples) < 100:
        raise ValueError(f"Not enough GPS samples found in {file} to locate track")

    mean_point = GPSSample(
        lat=np.mean([s.lat for s in samples]),
        lon=np.mean([s.lon for s in samples]),
        altitude=np.mean([s.altitude for s in samples]),
    )

    global cs
    if cs is None:
        cs = CoordinateSystem(mean_point)

    return mean_point

In [12]:
gdrive = Path(
    "/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive"
)
tracks_storage = gdrive / "karting" / "videos"
tracks_storage.mkdir(exist_ok=True, parents=True)

In [19]:
import dataclasses
import itertools


tracks: list[tuple[str, GPSSample]] = [
    ("Buckmore Park", GPSSample(lat=51.343, lon=0.502, altitude=0)),
    ("Daytona Sandown Park", GPSSample(lat=51.375755, lon=-0.361993, altitude=0)),
    ("Daytona Milton Keynes", GPSSample(lat=52.040988, lon=-0.784958, altitude=0)),
    ("Whilton Mill", GPSSample(lat=52.276616, lon=-1.088216, altitude=95.225930)),
    ("Clay Pigeon", GPSSample(lat=50.823994, lon=-2.555045, altitude=240.943227)),
    ("Rye House", GPSSample(lat=51.767428, lon=0.011796, altitude=26.470608)),
    ("Pro Kart Raceland", GPSSample(lat=49.326336, lon=12.214084, altitude=393.565381)),
]


def guess_track(location: GPSSample) -> str:
    closest_track = min(tracks, key=lambda t: cs.distance(t[1], location))
    if cs.distance(closest_track[1], location) < 10_000:
        return closest_track[0]
    else:
        name = f"Unknown track #{len(tracks)}"
        tracks.append((name, location))
        return name


by_track: dict[str, list[str]] = {}


@dataclasses.dataclass
class FolderWithVideos:
    path: Path
    forbidden: set[str] = dataclasses.field(default_factory=set)

    def __iter__(self):
        for p in self.path.glob("**/*.MP4"):
            if p.name not in self.forbidden:
                yield p


library = list(
    itertools.chain(
        FolderWithVideos(
            Path("/Users/denys/Documents/video-dump"),
            {
                "GH010236.MP4",  # cycle in the city
                "GH010237.MP4",  # cycle next to big ben
                "GH010238.MP4",  # cycle in the city
                "GH010239.MP4",  # cycle in the city
                "GH010240.MP4",  # cycle in the city
                "GH010241.MP4",  # cycle in the city
                "GH010242.MP4",  # black recording
                "GH010243.MP4",  # broken timestamps
            },
        ),
        FolderWithVideos(Path("/Users/denys/dev/pacer/videos/clay-pigeon")),
        FolderWithVideos(Path("/Users/denys/dev/pacer/videos/whilton-mill")),
        FolderWithVideos(Path("/Users/denys/Documents/2025-nov-buckmore-testing")),
        FolderWithVideos(Path("/Users/denys/Documents/2026-jan-buckmore-park")),
        # FolderWithVideos(Path("/Users/denys/Downloads")),
        FolderWithVideos(Path("/Users/denys/Desktop/Google Drive (Not synced)")),
        FolderWithVideos(gdrive / "photos/some-more-videos-backup"),
        FolderWithVideos(Path("/Users/denys/Documents/gokarting-ui")),
        FolderWithVideos(gdrive / "photos/gopro-backup"),
        FolderWithVideos(Path("/Volumes/Untitled/DCIM/100GOPRO")),
    )
)

In [20]:
import shutil


for p in library:
    try:
        print(f"Processing {p} ...")
        location = locate_track(str(p))
        track = guess_track(location)
        if not track.startswith("Unknown track"):
            (tracks_storage / track).mkdir(exist_ok=True)
            shutil.move(str(p), str(tracks_storage / track / p.name))
        by_track.setdefault(track, []).append(p)
        print(p.name, location)
    except Exception as e:
        print(p.name, "failed to locate track", e)

Processing /Users/denys/Documents/video-dump/GH010246.MP4 ...
GH010246.MP4 GPSSample(lat=51.360831, lon=0.029994, altitude=68.785463, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=0)
Processing /Users/denys/dev/pacer/videos/clay-pigeon/GX010128.MP4 ...
GX010128.MP4 failed to locate track Not enough GPS samples found in /Users/denys/dev/pacer/videos/clay-pigeon/GX010128.MP4 to locate track
Processing /Users/denys/dev/pacer/videos/clay-pigeon/GX010130.MP4 ...
GX010130.MP4 failed to locate track Not enough GPS samples found in /Users/denys/dev/pacer/videos/clay-pigeon/GX010130.MP4 to locate track
Processing /Users/denys/Documents/2025-nov-buckmore-testing/GH030279.MP4 ...
GH030279.MP4 failed to locate track Failed to open file: /Users/denys/Documents/2025-nov-buckmore-testing/GH030279.MP4
Processing /Users/denys/Documents/2025-nov-buckmore-testing/GH010279.MP4 ...
GH010279.MP4 GPSSample(lat=51.358531, lon=0.089909, altitude=54.054566, full_speed=0.000000, ground_speed=0.000000,

In [25]:
tracks

[('Buckmore Park',
  GPSSample(lat=51.343000, lon=0.502000, altitude=0.000000, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=0)),
 ('Daytona Sandown Park',
  GPSSample(lat=51.375755, lon=-0.361993, altitude=0.000000, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=0)),
 ('Daytona Milton Keynes',
  GPSSample(lat=52.040988, lon=-0.784958, altitude=0.000000, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=0)),
 ('Whilton Mill',
  GPSSample(lat=52.276616, lon=-1.088216, altitude=95.225930, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=0)),
 ('Clay Pigeon',
  GPSSample(lat=50.823994, lon=-2.555045, altitude=240.943227, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=0)),
 ('Rye House',
  GPSSample(lat=51.767428, lon=0.011796, altitude=26.470608, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=0)),
 ('Pro Kart Raceland',
  GPSSample(lat=49.326336, lon=12.214084, altitude=393.565381, full_speed=0.000000, ground_speed=0.000000, timestam

In [26]:
by_track['Unknown track #8']

[PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/some-more-videos-backup/GH010237.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH010237.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH020242.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH020237.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH010240.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH010241.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH010238.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH010239.MP4'),
 PosixPath('/

In [32]:
! open '/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH010239.MP4'

In [29]:
by_track['Unknown track #8']

[PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/some-more-videos-backup/GH010237.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH010237.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH020242.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH020237.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH010240.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH010241.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH010238.MP4'),
 PosixPath('/Users/denys/Library/CloudStorage/GoogleDrive-dendi239@gmail.com/My Drive/photos/gopro-backup/GH010239.MP4'),
 PosixPath('/

In [30]:
! open /Volumes/Untitled/DCIM/100GOPRO/GX010310.MP4

The file /Volumes/Untitled/DCIM/100GOPRO/GX010310.MP4 does not exist.


In [ ]:
{track: len(files) for track, files in by_track.items()}

{'Unknown track #7': 2,
 'Unknown track #8': 9,
 'Unknown track #9': 3,
 'Unknown track #10': 1,
 'Pro Kart Raceland': 2,
 'Unknown track #11': 1}

In [87]:
locate_track("/Users/denys/Documents/video-dump/GX010297.MP4")

GPSSample(lat=51.343060, lon=0.501149, altitude=109.616802, full_speed=0.000000, ground_speed=0.000000, timestamp_ms=0)

In [1]:
tracks

NameError: name 'tracks' is not defined

In [6]:
samples_df[["lat", "lon", "altitude"]].describe()

,lat,lon,altitude
count,509.000000,509.000000,509.000000
mean,51.343060,0.501149,109.616802
std,0.000350,0.000263,1.767304
min,51.342388,0.500508,105.144000
25%,51.342797,0.501105,108.721000
50%,51.343043,0.501131,110.477000
75%,51.343384,0.501310,110.669000
max,51.343589,0.501695,111.865000


In [3]:
mean_point = (
    (samples_df[["lat", "lon", "altitude"]].T * samples_df["full_speed"])
    .mean(axis=1)
    .pipe(lambda s: s / samples_df["full_speed"].mean())
    .pipe(lambda s: GPSSample(**s.to_dict()))
)

cs = CoordinateSystem(mean_point)

In [5]:
coords = pd.DataFrame(
    asdict(
        cs.local(GPSSample(lat=row["lat"], lon=row["lon"], altitude=row["altitude"]))
    )
    for _, row in samples_df.iterrows()
)

In [6]:
def rot(d: pd.DataFrame, angle: float) -> pd.DataFrame:
    x = d["x"]
    y = d["y"]
    new_x = x * np.cos(angle) - y * np.sin(angle)
    new_y = x * np.sin(angle) + y * np.cos(angle)
    return d.assign(x=new_x, y=new_y)

In [7]:
px.line(rot(coords, 0.4), x="x", y="y")

In [8]:
px.line(coords["z"].rolling(2000).mean())

In [ ]:
rot_coords = rot(coords, 0.4)
d_coords = rot_coords - rot_coords.mean()
(d_coords**2).mean(), (d_coords**2).mean().sum()

(x    4880.278291
 y    1441.220497
 z       1.320764
 dtype: float64,
 np.float64(6322.819552018278))

In [10]:
full_df = pd.concat([samples_df, coords], axis=1)

In [11]:
full_df

,lat,lon,altitude,full_speed,ground_speed,timestamp_ms,x,y,z
0,52.040289,-0.784666,80.145,0.20,0.166,1749900458000,-85.930742,70.313481,1.328863
1,52.040291,-0.784663,80.266,0.20,0.125,1749900458099,-85.759557,70.557689,1.449614
2,52.040291,-0.784659,80.339,0.15,0.017,1749900458199,-85.478819,70.590806,1.522468
3,52.040291,-0.784660,80.747,0.05,0.107,1749900458299,-85.533603,70.589509,1.929630
4,52.040291,-0.784662,81.131,0.13,0.127,1749900458399,-85.650012,70.577171,2.312842
...,...,...,...,...,...,...,...,...,...
26524,52.040381,-0.785117,79.243,0.01,0.031,1749903110399,-116.832264,80.578890,0.428101
26525,52.040381,-0.785117,79.246,0.03,0.018,1749903110499,-116.832264,80.567763,0.431095
26526,52.040381,-0.785117,79.255,0.02,0.053,1749903110599,-116.832264,80.567734,0.440076
26527,52.040381,-0.785117,79.273,0.05,0.031,1749903110699,-116.832265,80.545441,0.458040


In [14]:
px.line_3d(full_df, x="lat", y="lon", z="altitude")

In [13]:
asdict(samples[0])

{'lat': 52.0402888,
 'lon': -0.7846659,
 'altitude': 80.145,
 'full_speed': 0.2,
 'ground_speed': 0.166,
 'timestamp_ms': 1749900458000}

In [15]:
samples_df = (
    pd.DataFrame(asdict(s) | asdict(cs.local(s)) for s in samples)
    .assign(timestamp=lambda d: pd.to_datetime(d["timestamp_ms"], unit="ms"))
    .drop(columns="timestamp_ms")
)

samples_df

,lat,lon,altitude,full_speed,ground_speed,x,y,z,timestamp
0,52.040289,-0.784666,80.145,0.20,0.166,-85.930742,70.313481,1.328863,2025-06-14 11:27:38.000
1,52.040291,-0.784663,80.266,0.20,0.125,-85.759557,70.557689,1.449614,2025-06-14 11:27:38.099
2,52.040291,-0.784659,80.339,0.15,0.017,-85.478819,70.590806,1.522468,2025-06-14 11:27:38.199
3,52.040291,-0.784660,80.747,0.05,0.107,-85.533603,70.589509,1.929630,2025-06-14 11:27:38.299
4,52.040291,-0.784662,81.131,0.13,0.127,-85.650012,70.577171,2.312842,2025-06-14 11:27:38.399
...,...,...,...,...,...,...,...,...,...
26524,52.040381,-0.785117,79.243,0.01,0.031,-116.832264,80.578890,0.428101,2025-06-14 12:11:50.399
26525,52.040381,-0.785117,79.246,0.03,0.018,-116.832264,80.567763,0.431095,2025-06-14 12:11:50.499
26526,52.040381,-0.785117,79.255,0.02,0.053,-116.832264,80.567734,0.440076,2025-06-14 12:11:50.599
26527,52.040381,-0.785117,79.273,0.05,0.031,-116.832265,80.545441,0.458040,2025-06-14 12:11:50.699


In [16]:
fig = px.line(samples_df, x="x", y="y")
fig

In [17]:
samples_df[["x", "y", "z"]].mean()

x   -4.624076
y    4.429509
z    0.066850
dtype: float64

In [18]:
((samples_df[["x", "y", "z"]] - samples_df[["x", "y", "z"]].mean()) ** 2).mean()

x    3176.799881
y    3144.698907
z       1.320764
dtype: float64

In [19]:
rolling = (
    samples_df.assign(
        x=lambda d: d["x"] * d["ground_speed"],
        y=lambda d: d["y"] * d["ground_speed"],
    )[["x", "y", "ground_speed"]]
    .rolling(10_00)
    .mean()
    .assign(
        x=lambda d: d["x"] / d["ground_speed"],
        y=lambda d: d["y"] / d["ground_speed"],
    )
)

fig.add_trace(go.Scatter(x=rolling["x"], y=rolling["y"]))
fig

In [20]:
rolling = samples_df[["x", "y", "ground_speed"]].rolling(1000).mean()

fig.add_trace(go.Scatter(x=rolling["x"], y=rolling["y"]))
fig